## Data Ingestion

In [1]:
import os 

os.makedirs('../data', exist_ok=True)


In [2]:
### Document is a primary data structure in LangChain. It is used to represent a single piece of text data, along with any associated metadata. The Document class is part of the langchain_core module and can be used to create and manipulate documents in a structured way.

from langchain_core.documents import Document

document = Document(
    page_content="This is a test document.",
    metadata={"source": "test_source"}
)

document

Document(metadata={'source': 'test_source'}, page_content='This is a test document.')

In [3]:
### PDF loader is a utility class in LangChain that allows you to load PDF files and convert them into Document objects. It is part of the langchain_core module and can be used to extract text from PDF files for further processing.

from langchain_community.document_loaders import PyPDFLoader


document_pdf = PyPDFLoader("../data/Human_Digital_Twins_Maternal_Hypertension_Literature_Review.pdf").load()

document_pdf

/var/folders/5s/pkxbz81j3x97nyl48tfvm5tr0000gn/T/ipykernel_9072/3962646408.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
/Users/bytecorp/Learning/Learning_LLMS/data_ingestion/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Document(metadata={'producer': 'Microsoft® Word 2010', 'creator': 'Microsoft® Word 2010', 'creationdate': '2026-07-05T21:19:34+05:00', 'author': 'Un-named', 'moddate': '2026-07-05T21:19:34+05:00', 'source': '../data/Human_Digital_Twins_Maternal_Hypertension_Literature_Review.pdf', 'total_pages': 14, 'page': 0, 'page_label': '1'}, page_content='1 \nAbstract \nHuman Digital Twins (HDTs) — dynamic, data-driven virtual representations of individuals — \nhave emerged as an influential concept across manufacturing, occupational health, affective \ncomputing, and, increasingly, healthcare. This review synthesizes twenty papers spanning two \nrelated literatures: (1) the foundational architecture, definitions, and cross-domain applications of \nHDTs, and (2) digital health and predictive -modeling approaches specifically targeting \nhypertensive disorders of pregnancy (HDP) and p reeclampsia. We trace the evolution of HDT \ntheory from conceptual disambiguation and networking architecture thr

In [4]:
### Directory loader is a utility class in LangChain that allows you to load all files from a specified directory and convert them into Document objects. It is part of the langchain_community module and can be used to process multiple files in a directory for further analysis.

from langchain_community.document_loaders import DirectoryLoader , PyPDFLoader

documents = DirectoryLoader(
    "../data",
    glob = "*.pdf",
    loader_cls=PyPDFLoader
).load()

documents



[Document(metadata={'producer': 'Microsoft® Word 2010', 'creator': 'Microsoft® Word 2010', 'creationdate': '2026-07-05T21:19:34+05:00', 'author': 'Un-named', 'moddate': '2026-07-05T21:19:34+05:00', 'source': '../data/Human_Digital_Twins_Maternal_Hypertension_Literature_Review.pdf', 'total_pages': 14, 'page': 0, 'page_label': '1'}, page_content='1 \nAbstract \nHuman Digital Twins (HDTs) — dynamic, data-driven virtual representations of individuals — \nhave emerged as an influential concept across manufacturing, occupational health, affective \ncomputing, and, increasingly, healthcare. This review synthesizes twenty papers spanning two \nrelated literatures: (1) the foundational architecture, definitions, and cross-domain applications of \nHDTs, and (2) digital health and predictive -modeling approaches specifically targeting \nhypertensive disorders of pregnancy (HDP) and p reeclampsia. We trace the evolution of HDT \ntheory from conceptual disambiguation and networking architecture thr

### LangChain Text Splitter
- Chunking means splitting the document, we can't give whole document to AI for creating embedding that's why we need chunking 

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    length_function=len
)

chunks = splitter.split_documents(documents)

chunks 

[Document(metadata={'producer': 'Microsoft® Word 2010', 'creator': 'Microsoft® Word 2010', 'creationdate': '2026-07-05T21:19:34+05:00', 'author': 'Un-named', 'moddate': '2026-07-05T21:19:34+05:00', 'source': '../data/Human_Digital_Twins_Maternal_Hypertension_Literature_Review.pdf', 'total_pages': 14, 'page': 0, 'page_label': '1'}, page_content='1 \nAbstract \nHuman Digital Twins (HDTs) — dynamic, data-driven virtual representations of individuals — \nhave emerged as an influential concept across manufacturing, occupational health, affective \ncomputing, and, increasingly, healthcare. This review synthesizes twenty papers spanning two \nrelated literatures: (1) the foundational architecture, definitions, and cross-domain applications of \nHDTs, and (2) digital health and predictive -modeling approaches specifically targeting'),
 Document(metadata={'producer': 'Microsoft® Word 2010', 'creator': 'Microsoft® Word 2010', 'creationdate': '2026-07-05T21:19:34+05:00', 'author': 'Un-named', 'mo

In [6]:
import tiktoken

### in prouction systems we use token base chunking to avoid token limit issues with LLMs. The following code defines a function to count tokens in a given text using the tiktoken library, and then uses this function to create a RecursiveCharacterTextSplitter that splits documents based on token count.
def token_count(text: str) -> int:
    encoding = tiktoken.encoding_for_model('gpt-4')
    tokens = encoding.encode(text)
    return len(tokens)


token_base_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=20, 
    length_function=token_count
)   

result_token = splitter.split_documents(documents)




### Chroma and Vector DB

In [7]:
import uuid
from chromadb.config import Settings
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from typing import List, Dict, Any, Optional
from sklearn.metrics.pairwise import cosine_similarity
import os



In [8]:
class EmbeddingManager:
    def __init__(self , model_name : str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        try:
            self.model = SentenceTransformer(self.model_name)
            print("Model loaded successfully")
        except Exception as e:
            print(f"Error loading model: {e}")
            raise e
    # for creating embeddings for documents
    def embed_documents(self, documents: List[Document]) -> np.ndarray:
        try:
            texts = [document.page_content for document in documents]
            return self.model.encode(texts) 
        except Exception as e:
            print(f"Error embedding documents: {e}")
            raise e
    # for creating embeddings for queries
    def embed_query(self, query: str) -> np.ndarray:
        try:
            return self.model.encode(query)
        except Exception as e:
            print(f"Error embedding query: {e}")
            raise e
    
    
    
embedding_manager = EmbeddingManager()


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6165.19it/s]


Model loaded successfully


In [9]:
# adding the check for the length of the documents and the embeddings

if(len(chunks) == len(embedding_manager.embed_documents(chunks))):
    print(f"Documents and embeddings have the same length i.e {len(chunks)}")
else:
    print("Documents and embeddings have different lengths")


Documents and embeddings have the same length i.e 96


In [10]:
class VectorStore:
    def __init__(self, collection_name: str = "pdf_document_collection" , persistent_directory: str = "../data/vector_store"):
        self.collection_name = collection_name 
        self.persistent_directory = persistent_directory
        self.collection = None
        self.client = None
        self._create_vector_store()

    # creating vector store
    def _create_vector_store(self):
        try:
            #creating directory if not exists 
            os.makedirs(self.persistent_directory, exist_ok=True)
            #creating the persistent client so that we can store the vectro data in the disk , becuase we can't directly save the data in the memory
            self.client = chromadb.PersistentClient(path=self.persistent_directory)
            #creating the collection so that we can store the vector data in the collection 
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "This is a collection of documents for the vector store",
                "hnsw:space": "cosine"
                }
            )
        except Exception as e:
            print(f"Error creating vector store: {e}")

    def add_documents(self, documents: List[Document] , embedding : np.ndarray):
        try: 
            #adding the documents to the collection 
            if len(documents) == 0:
                raise ValueError("No documents to add")
            if len(embedding) == 0:
                raise ValueError("No embedding to add")
            if len(documents) != len(embedding):
                raise ValueError("Documents and embedding must have the same length")

            embeddings = embedding.tolist()
            metadatas = [document.metadata for document in documents]
            ids = [str(uuid.uuid4()) for _ in range(len(documents))]
            documents = [document.page_content for document in documents]
            
            #adding the documents to the collection 
            self.collection.add(
                ids=ids,
                documents=documents,
                metadatas=metadatas,
                embeddings=embeddings
            )
            print(f"Documents added to the vector store successfully")
        except Exception as e:
            print(f"Error adding documents to the vector store: {e}")
            raise e
    

vector_store = VectorStore()


In [11]:
# adding the documents to the vector store
vector_store.add_documents(chunks, embedding_manager.embed_documents(chunks))

Documents added to the vector store successfully


In [ ]:
# getting the collection data 
collection_data = vector_store.collection.get(include=["embeddings"])

collection_data

## retrievel phase getting the most relevant chunks from vector db


In [43]:
from typing import Any
class RetrieverManager:
    def __init__(self, vector_store: VectorStore , embedding_manager: EmbeddingManager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def get_relevant_chunks(self, query: str, k: int = 10 , threshold: float = 0.0) -> List[Dict[str, Any]]:
        # getting the query embedding
        query_embedding = self.embedding_manager.embed_query(query)
        # getting the most relevant chunks from the vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=k
            )
            retrieved_chunks = []
            results_documents = results["documents"][0]
            results_metadatas = results["metadatas"][0]
            results_distances = results["distances"][0]
            results_ids = results["ids"][0]

            for i , (results_document , results_metadata , results_distance , results_id) in enumerate(zip(results_documents , results_metadatas , results_distances , results_ids)):
                similarity_score = 1 - results_distance
                if similarity_score >= threshold:
                    retrieved_chunks.append({
                        "document" : results_document,
                        "metadata" : results_metadata,
                        "similarity_score" : similarity_score,
                        "id" : results_id,
                    })
            if(len(retrieved_chunks) == 0):
                print("No relevant chunks found")
                return []    
            print(f"Retrieved {len(retrieved_chunks)} chunks")

            return retrieved_chunks
        except Exception as e:
            print(f"Error getting relevant chunks: {e}")
    

retriever_manager = RetrieverManager(vector_store, embedding_manager)





## Integration of AI 

In [44]:
import os
from dotenv import load_dotenv
import langchain_groq
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate

load_dotenv()

query = """
Who is Abdul Karim Bukhsh Ansari?

Give a complete professional introduction including:
- Education
- Experience
- Skills
- Projects
- Technologies
"""
retrieved_chunks = retriever_manager.get_relevant_chunks(query)
relevant_context = "\n".join([chunk["document"] for chunk in retrieved_chunks])


llm = ChatGroq(
    model_name="llama-3.1-8b-instant",
    api_key=os.getenv("GROQ_API_KEY"),
    temperature=0.7,
    max_tokens=1000,
)

prompt = PromptTemplate(
    input_variables=["relevant_context", "query"],
    template="""
You are a helpful AI assistant.

Your task is to answer the user's question using ONLY the information provided in the context.

Rules:
1. Use ONLY the provided context. Do not use external knowledge.
2. If the answer is not present in the context, reply:
   "I don't know based on the provided context."
3. Always respond in the THIRD PERSON when referring to people mentioned in the context.
4. Never assume the identity of the person described in the context.
5. Never use first-person pronouns such as "I", "me", "my", or "mine" when describing the person in the context.
6. Treat the context as information about another person, not yourself.
7. Keep the answer concise, accurate, and professional.

Context:
{relevant_context}

Question:
{query}

Answer:
"""
)

chain = prompt | llm

response = chain.invoke({"relevant_context": relevant_context, "query": query})

print(response.content)



Retrieved 10 chunks
Abdul Karim Bukhsh Ansari is an Undergraduate Software Engineer with over 1 year of professional experience in the software industry. 

Education:
Abdul Karim Bukhsh Ansari is currently pursuing a Bachelor of Engineering in Software Engineering from NED University Of Engineering And Technology, scheduled to complete in August 2027.

Experience:
Abdul Karim Bukhsh Ansari has experience as an Associate Software Engineer at ByteCorp Technologies in Karachi, Pakistan, where he enhanced and optimized the company website, and developed a cross-platform mobile application for client NXTSTIM using React Native. He also worked as a Software Engineer Intern at Lambda Theta in Karachi, Pakistan, where he contributed to AI tools, writing clean and maintainable code, and communicating effectively with stakeholders.

Skills:
Abdul Karim Bukhsh Ansari is proficient in Languages & Frameworks such as JavaScript, Python, React.js, Next.js, Express, Nest.js, React Native Expo, SQL, No